# 基礎 LRU Cache

In [12]:
from collections import OrderedDict

cache = OrderedDict()
print(type(cache))

<class 'collections.OrderedDict'>


In [ ]:
from collections import OrderedDict

class SimpleLRUCache:
    def __init__(self, capacity: int):
        self.cache = OrderedDict()
        self.capacity = capacity

    def get(self, key: str) -> str | None: 
        """
        獲取資料。如果存在，將其移到最前面（表示最近使用）。
        """
        if key not in self.cache:
            return None
        
        # pop() 會移除並返回該項目，然後我們再把它放回去
        # 這樣就實現了「移到最前面」的效果
        value = self.cache.pop(key)
        self.cache[key] = value 
        return value

    def put(self, key: str, value: str) -> None:
        """
        新增或更新資料。
        """
        if key in self.cache:
            # 如果 key 已存在，先移除舊的
            self.cache.pop(key)
        elif len(self.cache) >= self.capacity:
            # 如果快取已滿，移除最久未使用的（最後一個）
            self.cache.popitem(last=False)
        
        # 將新項目放到最前面
        self.cache[key] = value

    def __str__(self):
        return str(self.cache)

In [ ]:
# --- 測試 ---
cache = SimpleLRUCache(capacity=3)

print(f"初始狀態: {cache}")

cache.put("A", "Data A")
cache.put("B", "Data B")
cache.put("C", "Data C")
print(f"放入 A, B, C 後: {cache}")

# 存取 A，使其變成最近使用
print(f"存取 A: {cache.get('A')}")
print(f"存取 A 後: {cache}")

# 放入 D，此時快取滿了，應該淘汰 B
cache.put("D", "Data D")
print(f"放入 D 後 (B 應被淘汰): {cache}")

# 存取 B，應該返回 None, and the current cache
print(f"存取 B: {cache.get('B')}")
print(f"Current cache: {cache}")

初始狀態: OrderedDict()
放入 A, B, C 後: OrderedDict({'A': 'Data A', 'B': 'Data B', 'C': 'Data C'})
存取 A: Data A
存取 A 後: OrderedDict({'B': 'Data B', 'C': 'Data C', 'A': 'Data A'})
放入 D 後 (B 應被淘汰): OrderedDict({'C': 'Data C', 'A': 'Data A', 'D': 'Data D'})
存取 B: None
Current cache: OrderedDict({'C': 'Data C', 'A': 'Data A', 'D': 'Data D'})


# ```OrderedDict``` vs ```@lru_cache``` 完整對比

---
## 1️⃣ 方法一：@lru_cache（Python 內建裝飾器）

**來源**：`functools.lru_cache`（Python 3.2+）

**特點**：
- 🎯 **函數級別快取**：只能快取「函數的返回值」
- 🔒 **參數限制**：函數參數必須是 hashable（不可變）
- 🚫 **無法快取物件**：不能快取 DataFrame、複雜物件
- ✅ **極簡單**：一行裝飾器搞定

In [ ]:
from functools import lru_cache
import time

# 範例：使用 @lru_cache 快取函數結果
@lru_cache(maxsize=128)
def fibonacci(n):
    """計算費波那契數列（經典範例）"""
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print("\n🔵 範例 1: @lru_cache 基礎用法\n")

# 第一次計算（Cache Miss）
start = time.time()
result1 = fibonacci(30)
time1 = time.time() - start
print(f"fibonacci(30) = {result1}")
print(f"第一次計算時間: {time1*1000:.2f}ms (Cache Miss)")

# 第二次計算（Cache Hit）
start = time.time()
result2 = fibonacci(30)
time2 = time.time() - start
print(f"第二次計算時間: {time2*1000:.6f}ms (Cache Hit)")
print(f"加速比: {time1/time2:.0f}x 倍\n")

# 查看快取資訊
print(f"快取統計: {fibonacci.cache_info()}")
print(f"  - hits: 快取命中次數")
print(f"  - misses: 快取未命中次數")
print(f"  - currsize: 當前快取大小")

### ❌ @lru_cache 的限制（為何不適合我們的專案）

**問題 1：參數必須是 hashable**

In [ ]:
print("\n🔴 範例 2: @lru_cache 的限制\n")

# 嘗試用 @lru_cache 快取 DataFrame 查詢（會失敗）
@lru_cache(maxsize=128)
def query_database_wrong(sql, params):
    """這個函數無法使用 @lru_cache！"""
    # params 可能是 list（不是 hashable）
    return f"Query result for: {sql}"

# 測試 1: hashable 參數（可以）
try:
    result = query_database_wrong("SELECT * FROM t", None)
    print(f"✅ 參數是 None（hashable）: 成功")
except TypeError as e:
    print(f"❌ 錯誤: {e}")

# 測試 2: list 參數（不可以）
try:
    result = query_database_wrong("SELECT * FROM t WHERE id=?", [5, 10])
    print(f"✅ 參數是 list: 成功")
except TypeError as e:
    print(f"❌ 參數是 list（unhashable）: {e}")

print("\n💡 解釋：")
print("   list, dict, DataFrame 都是 unhashable（可變物件）")
print("   @lru_cache 只能接受 tuple, str, int 等 hashable 參數")

## 2️⃣ 方法二：OrderedDict（自建 LRU Cache 類別）

**來源**：`collections.OrderedDict`（Python 3.1+）

**特點**：
- 🎯 **物件級別快取**：可以快取任何東西（DataFrame, 複雜物件）
- 🔓 **無參數限制**：可以處理 list, dict 等 unhashable 參數
- ✅ **完全控制**：TTL, 自訂驅逐策略, 監控指標
- 📊 **適合資料庫查詢**：這是我們專案的需求！

In [ ]:
import hashlib
import pandas as pd
from collections import OrderedDict

print("\n🟢 範例 3: OrderedDict 自建 LRU Cache（適合資料庫查詢）\n")

class QueryCache:
    """適合資料庫查詢的 LRU Cache"""
    def __init__(self, max_size=3):
        self.cache = OrderedDict()
        self.max_size = max_size
    
    def _generate_key(self, sql, params):
        """產生快取金鑰（可處理 unhashable 參數）"""
        params_str = str(params) if params else ""
        content = f"{sql}:{params_str}"
        return hashlib.md5(content.encode()).hexdigest()[:8]
    
    def get(self, sql, params):
        key = self._generate_key(sql, params)
        if key not in self.cache:
            return None
        self.cache.move_to_end(key)  # 標記為最近使用
        return self.cache[key]
    
    def set(self, sql, params, result):
        key = self._generate_key(sql, params)
        
        # 如果快取滿了，移除最舊項目
        if len(self.cache) >= self.max_size and key not in self.cache:
            oldest = next(iter(self.cache))
            del self.cache[oldest]
            print(f"   ❌ 快取滿，移除: {oldest}")
        
        self.cache[key] = result
        print(f"   ✅ 快取: {key} → {result}")

# 測試：模擬資料庫查詢
cache = QueryCache(max_size=3)

print("1️⃣ 查詢 I-5（參數是 list）")
cache.set("SELECT * WHERE route=?", [5], "DataFrame[I-5 data, 1000 rows]")

print("\n2️⃣ 查詢 I-405（參數是 list）")
cache.set("SELECT * WHERE route=?", [405], "DataFrame[I-405 data, 800 rows]")

print("\n3️⃣ 查詢 SR-91（參數是 list）")
cache.set("SELECT * WHERE route=?", [91], "DataFrame[SR-91 data, 600 rows]")

print("\n4️⃣ 再次查詢 I-5（Cache Hit！）")
result = cache.get("SELECT * WHERE route=?", [5])
print(f"   🎯 從快取取得: {result}")

print("\n5️⃣ 查詢 I-10（快取滿了，移除 I-405）")
cache.set("SELECT * WHERE route=?", [10], "DataFrame[I-10 data, 900 rows]")

print(f"\n📦 最終快取內容: {list(cache.cache.values())}")

## 📊 完整對比表（Comparison Table）

| 特性 | @lru_cache | OrderedDict (自建) | 本專案選擇 |
|------|-----------|-------------------|-----------|
| **使用難度** | ⭐⭐⭐⭐⭐ 極簡單（一行） | ⭐⭐⭐ 需自己實作 | OrderedDict |
| **參數限制** | ❌ 只支援 hashable | ✅ 無限制 | OrderedDict |
| **快取內容** | ❌ 只能快取函數返回值 | ✅ 任何物件（DataFrame） | OrderedDict |
| **TTL 支援** | ❌ 無內建支援 | ✅ 自己實作 | OrderedDict |
| **監控指標** | ⚠️ 基礎（cache_info）| ✅ 完全自訂 | OrderedDict |
| **驅逐策略** | ✅ LRU | ✅ 自訂（LRU/LFU/FIFO）| OrderedDict |
| **記憶體控制** | ⚠️ 只能設 maxsize | ✅ 完全控制 | OrderedDict |
| **跨方法使用** | ❌ 綁定單一函數 | ✅ 類別級別快取 | OrderedDict |

---

### 🎯 決策規則（Decision Rules）

#### **使用 @lru_cache 的場景**：
```python
# 1. 純函數（Pure Function）- 無副作用
@lru_cache(maxsize=128)
def calculate_distance(lat1, lon1, lat2, lon2):
    # 計算兩點距離（數學運算，無狀態）
    return math.sqrt((lat2-lat1)**2 + (lon2-lon1)**2)

# 2. 參數都是 hashable（int, str, tuple）
@lru_cache(maxsize=256)
def get_config(env: str, service: str):
    # 讀取配置檔案
    return load_config(env, service)

# 3. 簡單的遞迴優化（經典：費波那契、階乘）
@lru_cache
def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 1)
```

#### **使用 OrderedDict (自建) 的場景**：
```python
# 1. 資料庫查詢快取（我們的專案！）
class QueryEngine:
    def __init__(self):
        self.cache = QueryCache()  # OrderedDict 實作
    
    def execute(self, sql, params):  # params 可能是 list
        cached = self.cache.get(sql, params)
        if cached:
            return cached
        result = self.db.execute(sql, params)
        self.cache.set(sql, params, result)
        return result

# 2. 需要 TTL（快取過期）
class APICache:
    def __init__(self):
        self.cache = OrderedDict()
        self.timestamps = {}
        self.ttl = 300  # 5 分鐘過期

# 3. 需要自訂監控指標
class MetricsCache:
    def get_stats(self):
        return {
            'hit_rate': self.hits / (self.hits + self.misses),
            'avg_size': sum(sys.getsizeof(v) for v in self.cache.values()),
        }

# 4. 快取複雜物件（DataFrame, numpy array, 自訂類別）
cache.set(key, pd.DataFrame(...))  # ✅ 可以
cache.set(key, np.array(...))      # ✅ 可以
```

## 🏭 產業實務（Industry Practice）

### **Big Tech 公司的做法**

| 公司 | 快取策略 | 工具 | 場景 |
|------|---------|------|------|
| **Google** | 分散式快取 | Memcached, 自建 | BigQuery 查詢快取 |
| **Meta** | 混合快取 | EVCache (自建) | Presto 查詢結果 |
| **Netflix** | 多層快取 | EVCache + 本地 LRU | API 響應快取 |
| **Airbnb** | Application Cache | Ruby LRU + Redis | Metrics Dashboard |
| **Uber** | 自建快取 | Go LRU implementation | Query Parser |

**共同點**：
- ✅ **都使用自建 LRU Cache**（不用 @lru_cache）
- ✅ **支援 TTL 機制**（避免 stale data）
- ✅ **監控指標完整**（hit rate, latency, size）
- ✅ **支援複雜物件**（protobuf, JSON, DataFrame）

---

### **開源專案的做法**

#### **1. Django ORM（Python Web Framework）**
```python
# Django 的 cache framework（類似我們的實作）
from django.core.cache import cache

# 快取資料庫查詢結果（支援任何物件）
cache.set('user:123', user_obj, timeout=300)  # TTL 5 分鐘
user = cache.get('user:123')  # 可能是 None（過期或不存在）
```

#### **2. Flask-Caching（Python Web Framework）**
```python
from flask_caching import Cache

cache = Cache(config={'CACHE_TYPE': 'simple'})

# 也是自建 LRU，不用 @lru_cache
@cache.memoize(timeout=300)
def get_user_data(user_id):
    return db.query(...).all()  # 快取 DataFrame, list 等
```

#### **3. Pandas（Data Analysis Library）**
```python
# Pandas 內部也使用自建快取（不是 @lru_cache）
# 用於快取 read_csv, read_parquet 的 metadata
```

#### **4. DuckDB（我們使用的資料庫）**
```cpp
// DuckDB 內部的查詢計畫快取（C++ 實作）
class QueryCache {
    std::unordered_map<std::string, CachedPlan> cache;
    // 自建 LRU，不是標準庫的 std::lru_cache
};
```

---

### **為何產業不用 @lru_cache？**

1. **參數限制太嚴格**
   - 資料庫查詢參數常是 list, dict（unhashable）
   - @lru_cache 無法處理

2. **無法快取複雜物件**
   - DataFrame, numpy array, 自訂物件
   - 這些是資料工程的核心資料結構

3. **缺少 TTL 機制**
   - 資料會更新，快取需要過期
   - @lru_cache 沒有內建 TTL

4. **監控指標不足**
   - 生產環境需要詳細指標（hit rate, latency, memory）
   - @lru_cache.cache_info() 太簡單

5. **無法跨方法共享**
   - @lru_cache 綁定單一函數
   - 實際應用需要類別級別的快取

---

### **Industry SOP（產業標準作業程序）**

#### **Level 1: 小型專案/原型（Prototype）**
- ✅ 可以用 @lru_cache
- 場景：簡單函數、數學運算、配置讀取
- 範例：個人專案、MVP、腳本工具

#### **Level 2: 中型專案/生產系統（Production）**
- ✅ **自建 LRU Cache**（OrderedDict 或 dict + deque）
- 場景：資料庫查詢、API 快取、儀表板
- 範例：**我們的專案**、公司內部工具

#### **Level 3: 大型專案/分散式系統（Distributed）**
- ✅ 使用分散式快取（Redis, Memcached）
- 場景：微服務、高併發、跨伺服器共享
- 範例：Google, Meta, Netflix 級別

---

### **結論：為何我們選擇 OrderedDict？**

✅ **符合專案需求**：
1. 查詢參數是 list（@lru_cache 不支援）
2. 快取 DataFrame（@lru_cache 不支援）
3. 需要 TTL 機制（5 分鐘過期）
4. 需要監控指標（hit rate, cache size）

✅ **符合產業標準**：
1. Django, Flask, Pandas 都用自建快取
2. Google, Meta, Netflix 都用自建快取
3. 這是 **Mid-Senior DE/MLE 的必備技能**

✅ **展示技術深度**：
- 履歷亮點：「設計並實作 LRU Cache，支援 TTL 與監控指標」
- 面試價值：能回答「為何不用 @lru_cache？」
- 產業對應：Redis, Memcached 的核心概念